# Lab 55: Calibrated detection and judgment

Replace two stand-ins with calibration on held-out data: the fixed cosine threshold of [Lab 50](../50-closing-the-failure-loop/) (tune it by Youden's $J$) and the additive judge shift + all-dims gate of [Lab 51](../51-calibrated-multidimensional/) (isotonic calibration + a weighted gate). Math: [math-foundations/15](../../math-foundations/15-calibration-threshold-selection.md). Fill in the `TODO` cells; reference in `solution/`.

## Step 0: Setup

In [ ]:
from calibrate import (embed, cosine, make_change_pairs, tune_threshold,
                       qwk, additive_shift, isotonic_fit, isotonic_predict, weighted_gate)
# Two stand-ins replaced by calibration on held-out data: a fixed cosine threshold (Lab 50) and
# a single additive judge shift + all-dims gate (Lab 51). Math: math-foundations/15.
print("Part A: tune the change-detection threshold | Part B: isotonic judge calibration + weighted gate")

## Part A: tune the change-detection threshold (item 2)

In [ ]:
# TODO: embed the labeled pairs from make_change_pairs(), compute cosine per pair, and call
# tune_threshold(...). Compare the tuned threshold to the fixed 0.98 on accuracy and FPR. Why is
# the fixed cutoff a problem when reflow and changed cosines overlap?
raise NotImplementedError

## Part B: isotonic judge calibration (item 3)

In [ ]:
# TODO: with the compressed judge bias (gold 0,1,2,3 -> judge 0,0,1,2), fit an additive shift
# and an isotonic map on the calibration split, then compare completeness QWK on the test split:
# raw vs additive vs isotonic. Why can't the additive shift match isotonic here?
raise NotImplementedError

## Part B: a weighted multi-dimensional gate

In [ ]:
# A weighted multi-dimensional gate replaces all-dimensions-pass. Weights say which dimensions
# the product owner values; one strong dimension can offset a weaker one. This is a product
# decision, not a statistical one.
F=[3,3,2,1]; R=[3,2,3,1]; C=[1,2,1,3]; sbd={"f":F,"r":R,"c":C}; W={"f":0.5,"r":0.3,"c":0.2}
all_dims=lambda i: F[i]>=2 and R[i]>=2 and C[i]>=2
wgate=lambda i: weighted_gate(sbd,W,i,threshold=0.66)
for i in range(4):
    print(f"  release {i}: F{F[i]} R{R[i]} C{C[i]}  all-dims={'PASS' if all_dims(i) else 'BLOCK':5s}  weighted={'PASS' if wgate(i) else 'BLOCK'}")
print("\nRelease 0 (strong F/R, weak C) passes the weighted gate but fails all-dims - the gate now")
print("reflects that faithfulness matters more than completeness here.")

## What you built

Two stand-ins replaced by calibration on held-out data. **Part A**: the change detector's cosine threshold is tuned on labeled reflow/edit pairs by maximizing Youden's $J$, instead of a guessed 0.98. With real (overlapping) embedding distributions the fixed cutoff has a 0.33 false-positive rate - it flags a third of reflows as changes - while the tuned cutoff cuts that to 0.08 at higher accuracy. **Part B**: the judge's bias is monotone but not a constant offset, so an additive shift can't fix it; isotonic regression (pool-adjacent-violators) fits a monotone map judge -> gold and recovers more quadratic-weighted agreement (0.70 raw -> 0.88 additive -> 0.92 isotonic). And the all-dimensions-pass gate becomes a weighted gate, so a release strong on faithfulness can offset a weaker dimension - a product decision the weights make explicit.

**Where this simplifies:** the embedder is a deterministic char-trigram stand-in so the lab runs offline - production passes a sentence-transformer, and the *threshold-tuning procedure* is identical (label pairs, sweep, maximize $J$). PAVA is implemented inline to show the algorithm; `sklearn.isotonic.IsotonicRegression` is the production fit. The calibration and threshold are fit on a held-out split here too - refit them when the embedder or judge model changes, the same way you would retrain any model. The math is in [math-foundations/15](../../math-foundations/15-calibration-threshold-selection.md).